### 데이터셋 csv -> parquet 변환

In [2]:
import duckdb, os

In [3]:
con = duckdb.connect()
con.sql("""
    COPY (SELECT * FROM read_csv_auto('data/creditcard.csv'))
    TO 'data/creditcard.parquet' (FORMAT PARQUET)
""")
os.remove("data/creditcard.csv")
print("parquet 변환 완료")

parquet 변환 완료


# 온라인 결제 이상거래 탐지 (FDS) — EDA with SQL

## 프로젝트 배경 (risk management 기반 공부)

결제 서비스에서 이상거래탐지(Fraud Detection System)는 **사후 대응이 아닌 실시간 판단**의 영역이다.  
한 건의 사기를 놓치면 금전적 손실과 고객 신뢰가 동시에 무너지고,  
반대로 정상 거래를 과도하게 차단하면 고객 이탈과 매출 손실로 이어진다.

이 노트북에서는 Kaggle의 신용카드 사기 탐지 데이터셋을 사용해,  
**SQL 기반 탐색으로 이상거래의 패턴을 구조적으로 파악**하는 과정을 기록한다.

> **핵심 질문:** 거래 데이터만으로 사기를 의심할 수 있는 패턴은 무엇이며,  
> 그 패턴을 어떤 기준으로 규칙화할 수 있는가?

### 데이터 소개
- **출처:** [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- **구성:** 2013년 9월 유럽 카드홀더의 이틀간 거래 284,807건
- **피처:** PCA 변환된 V1~V28 (개인정보 보호), Time, Amount, Class(0=정상, 1=사기)
- **특이사항:** PCA 변환으로 원본 피처명을 알 수 없음 → 도메인 해석보다 **통계적 패턴 탐색**에 집중

### 분석 도구
- **DuckDB:** 로컬 환경에서 SQL로 대용량 데이터를 빠르게 탐색

In [ ]:
# !pip install duckdb pandas matplotlib seaborn

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [11]:
# 환경설정
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows
# plt.rcParams['font.family'] = 'AppleGothic'   # Mac
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 4)

In [6]:
# 데이터 로드 — parquet
# 최초 1회: CSV → Parquet 변환 후 사용 (로드 속도 약 10배 향상)
# con.execute("COPY (SELECT * FROM read_csv_auto('data/creditcard.csv')) TO 'data/creditcard.parquet' (FORMAT PARQUET)")

creditcard = con.execute("SELECT * FROM read_parquet('data/creditcard.parquet')").fetchdf()
print(f"데이터 shape: {creditcard.shape}")
print(f"메모리 사용량: {creditcard.memory_usage(deep=True).sum() / 1e6:.1f} MB")

데이터 shape: (284807, 31)
메모리 사용량: 70.6 MB


## 1. 전체 구조 파악

모델링이나 시각화 전, 데이터 확인

- **컬럼 구성과 타입**
- **결측치 유무**
- **기본 통계량**

특히 이 데이터는 PCA 변환된 익명 피처(V1~V28)가 대부분이므로,  
각 피처의 의미를 해석하기보다 **분포의 형태와 스케일 차이**에 주목하고자 함

In [10]:
con.execute("SELECT * FROM read_parquet('data/creditcard.parquet') LIMIT 5").fetchdf()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
# 컬럼 구성 및 타입 확인
con.execute("DESCRIBE SELECT * FROM read_parquet('data/creditcard.parquet')").fetchdf()

,column_name,column_type,null,key,default,extra
0,Time,BIGINT,YES,None,None,None
1,V1,DOUBLE,YES,None,None,None
2,V2,DOUBLE,YES,None,None,None
3,V3,DOUBLE,YES,None,None,None
4,V4,DOUBLE,YES,None,None,None
5,V5,DOUBLE,YES,None,None,None
6,V6,DOUBLE,YES,None,None,None
7,V7,DOUBLE,YES,None,None,None
8,V8,DOUBLE,YES,None,None,None
9,V9,DOUBLE,YES,None,None,None


In [8]:
# 결측치 확인
con.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(Time) AS time_null,
        COUNT(*) - COUNT(Amount) AS amount_null,
        COUNT(*) - COUNT(Class) AS class_null
    FROM read_parquet('data/creditcard.parquet')
""").fetchdf()

,total_rows,time_null,amount_null,class_null
0,284807,0,0,0


**결측치 : 0** 
이 데이터셋은 Kaggle에서 전처리된 상태로 제공되었으므로 결측치가 없는 걸로 보인다.
실무에서는 결측 자체가 이상 신호일 수 있음 (ex. 의도적으로 필드를 비운 거래)

### 타겟 변수(Class)의 분포 확인 _클래스 불균형 확인
 
평가지표 선택, 샘플링 전략, 임계값 설정을 위함.

In [9]:
# 타겟 분포: 정상 vs 사기
result = con.execute("""
    SELECT 
        Class,
        COUNT(*) AS cnt,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 4) AS pct
    FROM read_parquet('data/creditcard.parquet')
    GROUP BY Class
    ORDER BY Class
""").fetchdf()

print(result)
print(f"\n정상:사기 비율 = 1:{result['cnt'].iloc[0] / result['cnt'].iloc[1]:.0f}")

   Class     cnt      pct
0      0  284315  99.8273
1      1     492   0.1727

정상:사기 비율 = 1:578


사기 거래는 전체의 약 0.17%, 정상 대비 약 **1:578** 비율이다.

이 수준의 불균형은 두 가지를 의미한다:

1. **Accuracy는 무의미하다.** 모든 거래를 정상으로 찍어도 99.83%가 나온다.  
   → 평가지표는 **AUPRC(Precision-Recall AUC)** 또는 Recall 중심으로 설계해야 한다.

2. **False Positive를 어떻게 다룰 것인가가 핵심 설계 질문이다.**  
   Recall을 높이면 FP가 늘어나는데, FP를 단순히 "오탐"으로 버리지 않고  
   **"수동 검토 대기열(review queue)"로 활용**하면 실무적으로 의미가 있다고 함

## 2. Amount와 Time — 해석 가능한 피처 탐색

V1~V28은 PCA 변환으로 의미를 알 수 없다.  
도메인 관점에서 직접 해석할 수 있는 피처는 **거래 금액(Amount)** 과 **거래 시점(Time)** 두 개뿐이다.

여기서 던지는 질문:
- 사기 거래의 금액 분포는 정상 거래와 다른가?
- 사기가 집중되는 시간대가 있는가?